In [2]:
import pandas as pd
import numpy as np
import json
import os
from huggingface_hub import hf_hub_download
from typing import Dict, List, Any

# -----------------------------------------------------------------------------
# 1. 설정 (Config)
# -----------------------------------------------------------------------------
DATASET_ID = "RZ412/EmbedLLM"
SPLIT = "train"       # train, val, test 중 선택
USE_COST = True       # 비용 포함 여부
CHUNKSIZE = 200_000   # CSV 읽기 청크 사이즈

# 로컬 레지스트리 파일 경로 (사용자 지정 경로)
REGISTRY_PATH = "/home/sjy990426/Desktop/LLM_Router/Queing_MNL_Router/registry-embedllm.json"

# -----------------------------------------------------------------------------
# 2. 레지스트리(비용 정보) 로드
# -----------------------------------------------------------------------------
print(f"📂 Loading registry from {REGISTRY_PATH}...")

try:
    with open(REGISTRY_PATH, "r", encoding="utf-8") as f:
        registry = json.load(f)
    print(f"✅ 레지스트리 로드 완료 (총 {len(registry)}개 모델 정보)")
except FileNotFoundError:
    print(f"❌ 파일을 찾을 수 없습니다: {REGISTRY_PATH}")
    raise
except json.JSONDecodeError:
    print(f"❌ JSON 파싱 오류: {REGISTRY_PATH}")
    raise

# -----------------------------------------------------------------------------
# 3. 데이터 로드 및 피벗 (Pivot) 로직
# -----------------------------------------------------------------------------
print(f"🔄 Downloading/Loading metadata from {DATASET_ID}...")

# 3-1. 모델 목록 로드 (model_order.csv)
try:
    model_order_path = hf_hub_download(repo_id=DATASET_ID, filename="model_order.csv", repo_type="dataset")
    model_df = pd.read_csv(model_order_path)
    
    if "model_name" not in model_df.columns:
        raise ValueError("[EmbedLLM] model_order.csv에 'model_name' 컬럼이 없습니다.")
        
    models = model_df["model_name"].astype(str).tolist()
    K = len(models)
    model_set = set(models)
    
    print(f"✅ 감지된 모델({K}개): {models}")
    
except Exception as e:
    print(f"❌ 모델 목록 로드 실패: {e}")
    raise e

# 3-2. 메인 데이터 파일 다운로드
split2file = {"train": "train.csv", "validation": "val.csv", "val": "val.csv", "test": "test.csv"}
filename = split2file.get(SPLIT, SPLIT)
csv_path = hf_hub_download(repo_id=DATASET_ID, filename=filename, repo_type="dataset")

print(f"🔄 Processing {filename} (Chunk processing)...")

# 3-3. 청크 단위로 읽어서 Pivot 수행 (Long -> Wide)
store: Dict[int, Dict[str, Any]] = {}
completed: List[int] = []

reader = pd.read_csv(csv_path, chunksize=int(CHUNKSIZE))

for i, chunk in enumerate(reader):
    # 필수 컬럼 확인 (첫 번째 청크에서만)
    if i == 0:
        required_cols = ["prompt_id", "model_name", "label", "prompt"]
        if not all(col in chunk.columns for col in required_cols):
            raise ValueError(f"필수 컬럼 누락: {required_cols} 중 일부가 없습니다.")

    for r in chunk.itertuples(index=False):
        pid = int(getattr(r, "prompt_id"))
        mname = str(getattr(r, "model_name"))
        
        # 정의된 모델 목록에 없는 모델은 무시
        if mname not in model_set:
            continue

        # 저장소 초기화
        rec = store.get(pid)
        if rec is None:
            rec = {
                "prompt": str(getattr(r, "prompt")),
                "labels": {},
                "cnt": 0,
            }
            store[pid] = rec

        # 점수 기록
        labels = rec["labels"]
        if mname not in labels:
            labels[mname] = float(getattr(r, "label"))
            rec["cnt"] += 1
            
            # 모든 모델(K개)의 점수가 다 모였으면 완료 리스트에 추가
            if rec["cnt"] == K:
                completed.append(pid)

# -----------------------------------------------------------------------------
# 4. DataFrame 생성 (비용 매핑 수정됨)
# -----------------------------------------------------------------------------
rows = []
missing_cost_models = set()

for pid in completed:
    rec = store.get(pid)
    if rec is None or rec["cnt"] != K:
        continue
        
    row = {
        "sample_id": pid,
        "prompt": rec["prompt"],
        "eval_name": "embedllm",
        "oracle_model_to_route_to": "",
    }
    
    # 모델별 점수 및 비용 추가
    for m in models:
        # 1. 점수 (Label)
        row[m] = float(rec["labels"][m])
        
        # 2. 비용 (Cost) - JSON 파일에서 조회
        if USE_COST:
            # JSON 키와 CSV 모델명이 일치한다고 가정
            if m in registry:
                cost_val = registry[m].get("cost_usd_per_1k_tokens", 0.0)
            else:
                cost_val = 0.0
                missing_cost_models.add(m)
            
            # 요청하신 형식: "모델명|total_cost"
            row[f"{m}|total_cost"] = float(cost_val)
            
    rows.append(row)

df = pd.DataFrame(rows).reset_index(drop=True)

if missing_cost_models:
    print(f"⚠️ 주의: 다음 모델들은 JSON 레지스트리에 없어 비용이 0으로 처리되었습니다: {missing_cost_models}")

if len(df) == 0:
    print("⚠️ 경고: 완성된 데이터가 0건입니다.")
else:
    print(f"✅ 최종 데이터프레임 생성 완료: {len(df)} 행")

# -----------------------------------------------------------------------------
# 5. CSV 저장
# -----------------------------------------------------------------------------
output_filename = "embedllm_dataset_processed.csv"
df.to_csv(output_filename, index=False, encoding='utf-8-sig')

print(f"🎉 '{output_filename}' 파일로 저장이 완료되었습니다.")
print("-" * 30)

/home/sjy990426/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


📂 Loading registry from /home/sjy990426/Desktop/LLM_Router/Queing_MNL_Router/registry-embedllm.json...
✅ 레지스트리 로드 완료 (총 115개 모델 정보)
🔄 Downloading/Loading metadata from RZ412/EmbedLLM...
✅ 감지된 모델(112개): ['Qwen__Qwen1.5-7B-Chat', 'ConvexAI__Luminex-34B-v0.1', 'lmsys__vicuna-13b-v1.5', 'deepseek-ai__deepseek-math-7b-instruct', 'TigerResearch__tigerbot-13b-base', 'ConvexAI__Luminex-34B-v0.2', 'berkeley-nest__Starling-LM-7B-alpha', 'EleutherAI__llemma_7b', 'CultriX__NeuralTrix-bf16', 'SciPhi__SciPhi-Mistral-7B-32k', 'TheBloke__tulu-30B-fp16', 'lmsys__vicuna-33b-v1.3', 'scb10x__typhoon-7b', 'mlabonne__AlphaMonarch-7B', 'mistralai__Mistral-7B-Instruct-v0.1', '01-ai__Yi-34B-Chat', 'meta-llama__Llama-2-13b-chat-hf', 'eren23__ogno-monarch-jaskier-merge-7b-OH-PREF-DPO', 'ibivibiv__alpaca-dragon-72b-v1', 'golaxy__gowizardlm', 'codellama__CodeLlama-34b-Instruct-hf', 'OpenBuddy__openbuddy-codellama2-34b-v11.1-bf16', 'deepseek-ai__deepseek-coder-1.3b-base', 'Neko-Institute-of-Science__pygmalion-7b', 

In [8]:
import pandas as pd
from typing import Tuple, List, Dict

# -----------------------------------------------------------------------------
# 1. 헬퍼 함수 정의 (제공해주신 스크립트 로직 복원)
# -----------------------------------------------------------------------------
def infer_models_and_cost_map(df: pd.DataFrame) -> Tuple[List[str], Dict[str, str]]:
    base = {"sample_id", "prompt", "eval_name", "oracle_model_to_route_to"}
    # '|'가 없고 base 컬럼이 아닌 것들을 모델 컬럼으로 간주
    models = [c for c in df.columns if ("|" not in c) and (c not in base)]
    # '|total_cost'로 끝나는 컬럼을 비용 컬럼으로 간주하고 매핑 생성
    cost_map = {c.split("|")[0]: c for c in df.columns if c.endswith("|total_cost")}
    return models, cost_map

# -----------------------------------------------------------------------------
# 2. 실행 설정
# -----------------------------------------------------------------------------
input_path = "/home/sjy990426/Desktop/LLM_Router/Queing_MNL_Router/routerbench_0shot.pkl"
output_csv_path = "routerbench_0shot.csv"
use_cost = True  # 비용 데이터 포함 여부 (Config 대신 직접 설정)

# -----------------------------------------------------------------------------
# 3. 데이터 로드 및 변환
# -----------------------------------------------------------------------------
if __name__ == "__main__":
    try:
        print(f"🔄 Reading {input_path}...")
        df = pd.read_pickle(input_path)
        
        # 필수 컬럼 검사
        if "prompt" not in df.columns:
            raise ValueError("DataFrame에 'prompt' 컬럼이 없습니다.")

        # 모델 및 비용 컬럼 추론
        models, cost_map_all = infer_models_and_cost_map(df)
        print(f"✅ 감지된 모델({len(models)}개): {models}")

        # 저장할 컬럼 리스트 구성
        keep = []
        
        # 1) 메타 데이터
        for c in ["sample_id", "prompt", "eval_name", "oracle_model_to_route_to"]:
            if c in df.columns:
                keep.append(c)

        # 2) 모델 점수
        keep += models

        # 3) 비용 정보 (옵션)
        if use_cost:
            for m in models:
                if m in cost_map_all:
                    keep.append(cost_map_all[m])

        # 중복 제거 (순서 유지)
        seen = set()
        keep = [c for c in keep if not (c in seen or seen.add(c))]

        # 최종 DataFrame 생성
        df_filtered = df[keep].copy()
        
        # -----------------------------------------------------------------------------
        # 4. CSV 저장
        # -----------------------------------------------------------------------------
        df_filtered.to_csv(output_csv_path, index=False, encoding='utf-8-sig')
        print(f"\n🎉 변환 완료! 저장된 파일: {output_csv_path}")
        
        # 결과 미리보기
        print("-" * 30)
        print(df_filtered.head(2).T)

    except FileNotFoundError:
        print(f"❌ 파일을 찾을 수 없습니다: {input_path}")
        print("경로를 확인하거나 파일을 업로드해주세요.")
    except Exception as e:
        print(f"❌ 오류 발생: {e}")

🔄 Reading /home/sjy990426/Desktop/LLM_Router/Queing_MNL_Router/routerbench_0shot.pkl...
✅ 감지된 모델(11개): ['WizardLM/WizardLM-13B-V1.2', 'claude-instant-v1', 'claude-v1', 'claude-v2', 'gpt-3.5-turbo-1106', 'gpt-4-1106-preview', 'meta/code-llama-instruct-34b-chat', 'meta/llama-2-70b-chat', 'mistralai/mistral-7b-chat', 'mistralai/mixtral-8x7b-chat', 'zero-one-ai/Yi-34B-Chat']

🎉 변환 완료! 저장된 파일: routerbench_0shot.csv
------------------------------
                                                                                              0  \
sample_id                                                       Chinese_character_riddles.dev.0   
prompt                                        ['猜字谜，根据我给的描述猜出一个字(请从汉字的字形、发音、意义以及字的拆分组合等角度考虑)...   
eval_name                                                             Chinese_character_riddles   
oracle_model_to_route_to                                                       no_model_correct   
WizardLM/WizardLM-13B-V1.2                                  

In [6]:
import pandas as pd
import json
import os
from datasets import load_dataset
from typing import List, Dict, Any, Optional, Tuple, Set

# -----------------------------------------------------------------------------
# 1. 설정 (Config)
# -----------------------------------------------------------------------------
DATASET_ID = "llm-blender/mix-instruct"
SPLIT = "train" 
METRIC = "bertscore" 

MAX_ROWS = None
USE_COST = True  

INFER_COMMON_MODELS = True
INFER_COMMON_N = 2000

REGISTRY_PATH = "/home/sjy990426/Desktop/LLM_Router/Queing_MNL_Router/registry-mix-instruct.json"

# -----------------------------------------------------------------------------
# 2. 레지스트리 로드
# -----------------------------------------------------------------------------
print(f"📂 Loading registry from {REGISTRY_PATH}...")
try:
    with open(REGISTRY_PATH, "r", encoding="utf-8") as f:
        registry = json.load(f)
    print(f"✅ 레지스트리 로드 완료 (총 {len(registry)}개 모델 정보)")
except Exception as e:
    print(f"❌ 레지스트리 로드 실패: {e}")
    raise e

# -----------------------------------------------------------------------------
# 3. 헬퍼 함수
# -----------------------------------------------------------------------------
def _build_prompt(ex: Dict[str, Any]) -> str:
    if "prompt" in ex and ex["prompt"] is not None: return str(ex["prompt"])
    instr = str(ex.get("instruction", "") or "")
    inp = str(ex.get("input", "") or "")
    if instr and inp.strip(): return instr + "\n\n" + inp
    if instr: return instr
    if inp: return inp
    return str(ex.get("text", "") or "")

def _infer_common_models(ds, n_scan: int = 2000) -> List[str]:
    common: Optional[Set[str]] = None
    n = min(int(n_scan), len(ds))
    for i in range(n):
        ex = ds[i]
        candidates = ex.get("candidates", []) or []
        ms = {str(c.get("model")) for c in candidates if c.get("model") is not None}
        if not ms: continue
        common = ms if common is None else (common & ms)
    if not common: raise ValueError("공통 모델을 찾을 수 없습니다.")
    return sorted(list(common))

# -----------------------------------------------------------------------------
# 4. 데이터 로드 및 처리
# -----------------------------------------------------------------------------
print(f"🔄 Loading dataset: {DATASET_ID} ({SPLIT})...")
try:
    ds = load_dataset(DATASET_ID, split=SPLIT)
except Exception as e:
    print(f"❌ 데이터셋 로드 실패: {e}")
    raise e

# 모델 리스트 결정
if INFER_COMMON_MODELS:
    print(f"🔍 상위 {INFER_COMMON_N}개 샘플을 스캔하여 공통 모델을 찾습니다...")
    models = _infer_common_models(ds, n_scan=INFER_COMMON_N)
else:
    candidates0 = ds[0].get("candidates", []) or []
    models = sorted({str(c.get("model")) for c in candidates0 if c.get("model") is not None})

print(f"✅ 감지된 모델 목록 ({len(models)}개): {models}")
model_set = set(models)

# -----------------------------------------------------------------------------
# [핵심 수정] 모델 이름 매핑 (Manual Mapping)
# -----------------------------------------------------------------------------
print("🔄 모델 이름 매칭 및 비용 로드 중...")

# 1. 룩업 테이블 생성 (Full Key & Short Key)
registry_lookup = {}
for reg_key, reg_data in registry.items():
    cost = reg_data.get("cost_usd_per_1k_tokens", 0.0)
    # Case A: Full Key (e.g., mosaicml__mpt-7b-instruct)
    registry_lookup[reg_key] = cost
    # Case B: Short Key (e.g., mpt-7b-instruct)
    if "__" in reg_key:
        short_name = reg_key.split("__")[-1]
        registry_lookup[short_name] = cost

# 2. 수동 매핑 (데이터셋 이름 -> 레지스트리 키 또는 숏네임)
#    여기에 "데이터셋 이름": "레지스트리에서 찾을 수 있는 이름" 을 적어줍니다.
MANUAL_MAP = {
    "mpt-7b": "mpt-7b-instruct",       # mpt-7b는 instruct 가격 사용
    "vicuna-13b-1.1": "vicuna-13b-1.1", # 이름이 같지만 혹시 몰라 명시
    "alpaca-native": "alpaca-native",
    "chatglm-6b": "chatglm-6b",
    "dolly-v2-12b": "dolly-v2-12b",
    "flan-t5-xxl": "flan-t5-xxl",
    "koala-7B-HF": "koala-7B-HF",
    "llama-7b-hf-baize-lora-bf16": "llama-7b-hf-baize-lora-bf16",
    "moss-moon-003-sft": "moss-moon-003-sft",
    "oasst-sft-4-pythia-12b-epoch-3.5": "oasst-sft-4-pythia-12b-epoch-3.5",
    "stablelm-tuned-alpha-7b": "stablelm-tuned-alpha-7b"
}

model_costs = {}
missing_models = []

for m in models:
    cost_found = False
    
    # 전략 1: 레지스트리 룩업에서 직접 찾기 (Full Name or Short Name)
    if m in registry_lookup:
        model_costs[m] = registry_lookup[m]
        cost_found = True
        
    # 전략 2: 수동 매핑(MANUAL_MAP)을 통해 찾기
    elif m in MANUAL_MAP:
        mapped_name = MANUAL_MAP[m]
        if mapped_name in registry_lookup:
            model_costs[m] = registry_lookup[mapped_name]
            cost_found = True
    
    if not cost_found:
        model_costs[m] = 0.0
        missing_models.append(m)

if missing_models:
    print(f"⚠️ 주의: 다음 모델들은 매칭 실패 (비용 0 처리): {missing_models}")
else:
    print("✅ 모든 모델의 비용 정보를 성공적으로 찾았습니다!")

# -----------------------------------------------------------------------------
# 데이터 변환 Loop
# -----------------------------------------------------------------------------
rows = []
kept = 0
seen = 0

print("🔄 데이터 변환 중 (Chunk processing)...")

for idx, ex in enumerate(ds):
    seen += 1
    if MAX_ROWS is not None and kept >= int(MAX_ROWS):
        break

    candidates = ex.get("candidates", []) or []
    if not candidates: continue

    prompt = _build_prompt(ex)
    
    perf = {}
    
    for c in candidates:
        m = c.get("model", None)
        if m is None: continue
        m = str(m)
        if m not in model_set: continue

        scores = c.get("scores", {}) or {}
        if METRIC not in scores: continue
        
        perf[m] = float(scores[METRIC])

    if len(perf) != len(models): 
        continue

    row = {
        "sample_id": ex.get("id", idx),
        "prompt": prompt,
        "eval_name": "mix-instruct",
        "oracle_model_to_route_to": "",
    }
    
    for m in models:
        row[m] = perf[m]
        if USE_COST:
            row[f"{m}|total_cost"] = model_costs[m]

    rows.append(row)
    kept += 1
    
    if seen % 5000 == 0:
        print(f"   ...processed {seen} rows (kept: {kept})")

df = pd.DataFrame(rows)

need_cols = ["prompt"] + models
if USE_COST:
    need_cols += [f"{m}|total_cost" for m in models]

df = df.dropna(subset=need_cols).reset_index(drop=True)

if len(df) == 0:
    print("⚠️ 경고: 유효한 데이터가 0건입니다.")
else:
    print(f"✅ 데이터 변환 완료: 총 {len(df)} 행 (Scanned: {seen})")

file_name = "mixinstruct_dataset_processed.csv"
df.to_csv(file_name, index=False, encoding='utf-8-sig')

print(f"🎉 '{file_name}' 파일로 저장이 완료되었습니다.")
print("-" * 30)

📂 Loading registry from /home/sjy990426/Desktop/LLM_Router/Queing_MNL_Router/registry-mix-instruct.json...
✅ 레지스트리 로드 완료 (총 11개 모델 정보)
🔄 Loading dataset: llm-blender/mix-instruct (train)...
🔍 상위 2000개 샘플을 스캔하여 공통 모델을 찾습니다...
✅ 감지된 모델 목록 (12개): ['alpaca-native', 'chatglm-6b', 'dolly-v2-12b', 'flan-t5-xxl', 'koala-7B-HF', 'llama-7b-hf-baize-lora-bf16', 'moss-moon-003-sft', 'mpt-7b', 'mpt-7b-instruct', 'oasst-sft-4-pythia-12b-epoch-3.5', 'stablelm-tuned-alpha-7b', 'vicuna-13b-1.1']
🔄 모델 이름 매칭 및 비용 로드 중...
✅ 모든 모델의 비용 정보를 성공적으로 찾았습니다!
🔄 데이터 변환 중 (Chunk processing)...
   ...processed 5000 rows (kept: 5000)
   ...processed 10000 rows (kept: 10000)
   ...processed 15000 rows (kept: 15000)
   ...processed 20000 rows (kept: 20000)
   ...processed 25000 rows (kept: 25000)
   ...processed 30000 rows (kept: 30000)
   ...processed 35000 rows (kept: 35000)
   ...processed 40000 rows (kept: 40000)
   ...processed 45000 rows (kept: 45000)
   ...processed 50000 rows (kept: 50000)
   ...processed 55000 

In [7]:
df.head()


,sample_id,prompt,eval_name,oracle_model_to_route_to,alpaca-native,alpaca-native|total_cost,chatglm-6b,chatglm-6b|total_cost,dolly-v2-12b,dolly-v2-12b|total_cost,...,mpt-7b,mpt-7b|total_cost,mpt-7b-instruct,mpt-7b-instruct|total_cost,oasst-sft-4-pythia-12b-epoch-3.5,oasst-sft-4-pythia-12b-epoch-3.5|total_cost,stablelm-tuned-alpha-7b,stablelm-tuned-alpha-7b|total_cost,vicuna-13b-1.1,vicuna-13b-1.1|total_cost
0,unified_chip2/83622,I want to get a tattoo but I'm not sure what k...,mix-instruct,,0.724643,0.00156,0.685458,0.00072,0.636591,0.00144,...,0.683767,0.00084,0.624464,0.00084,0.725064,0.00144,0.671663,0.00084,0.593226,0.00156
1,itwgpt4/34960,"Given the following context, suggest a gift id...",mix-instruct,,0.696609,0.00156,0.678135,0.00072,0.574656,0.00144,...,0.584543,0.00084,0.648243,0.00084,0.715251,0.00144,0.568700,0.00084,0.725819,0.00156
2,itwgpt4/20893,"Given a complicated sentence, rewrite it in a ...",mix-instruct,,0.751214,0.00156,0.798641,0.00072,0.530012,0.00144,...,0.548901,0.00084,0.560887,0.00084,0.774326,0.00144,0.570186,0.00084,0.599043,0.00156
3,unified_chip2/128676,Make a concise location description of a eerie...,mix-instruct,,0.713467,0.00156,0.709021,0.00072,0.579177,0.00144,...,0.550763,0.00084,0.589191,0.00084,0.842536,0.00144,0.705799,0.00084,0.772301,0.00156
4,unified_chip2/113627,Python function to change input to upper case.,mix-instruct,,0.664963,0.00156,0.744671,0.00072,0.693045,0.00144,...,0.608857,0.00084,0.601350,0.00084,0.680201,0.00144,0.673281,0.00084,0.684274,0.00156


In [11]:
import pandas as pd
import re
from datasets import load_dataset
from typing import Dict, Tuple, List, Any, Optional

# -----------------------------------------------------------------------------
# 1. 설정
# -----------------------------------------------------------------------------
DATASET_ID = "CARROT-LLM-Routing/SPROUT-o3mini"
SPLIT = "train"
OUTPUT_CSV = "sprout_o3mini_routerbench_like.csv"

# -----------------------------------------------------------------------------
# 2. 가격표 (Table 2, $ per 1M tokens) - 표준 키
# -----------------------------------------------------------------------------
SPROUT_PRICE_1M: Dict[str, Tuple[float, float]] = {
    "openai-o3-mini": (1.1, 4.4),
    "claude-3-5-sonnet-v1": (3.0, 15.0),
    "titan-text-premier-v1": (0.5, 1.5),
    "openai-gpt-4o": (2.5, 10.0),
    "openai-gpt-4o-mini": (0.15, 0.6),
    "openai-o1-mini": (1.1, 4.4),
    "granite-3-2b-instruct": (0.1, 0.1),
    "granite-3-8b-instruct": (0.2, 0.2),
    "llama-3-1-70b-instruct": (0.9, 0.9),
    "llama-3-1-8b-instruct": (0.2, 0.2),
    "llama-3-2-1b-instruct": (0.06, 0.06),
    "llama-3-2-3b-instruct": (0.06, 0.06),
    "llama-3-3-70b-instruct": (0.9, 0.9),
    "mixtral-8x7b-instruct": (0.6, 0.6),
    "llama-3-405b-instruct": (3.5, 3.5),
}

# -----------------------------------------------------------------------------
# 3. 유연한 매핑 로직 (The "Yudori" Logic)
# -----------------------------------------------------------------------------
def match_price_key(col_name: str) -> Optional[str]:
    """
    데이터셋의 복잡한 컬럼명을 가격표의 단순한 키와 매칭합니다.
    예: 'aws-claude-3-5-sonnet-v1' -> 'claude-3-5-sonnet-v1'
        'wxai-granite-3-8b-instruct-8k-max-tokens' -> 'granite-3-8b-instruct'
    """
    # 1. 비교를 위해 소문자 변환
    s = col_name.lower()

    # 2. 알려진 벤더 접두사 제거 (wxai-, aws-)
    #    openai-는 가격표 키에 포함되어 있으므로 제거하지 않거나, 제거 후 다시 붙임
    for prefix in ["wxai-", "aws-"]:
        if s.startswith(prefix):
            s = s[len(prefix):]
    
    # 3. 불필요한 접미사(Suffix) 제거 (Regex 사용)
    #    -8k-max-tokens, -v01, -v0.1 등 제거
    #    순서 중요: 긴 것부터 제거
    s = re.sub(r"-\d+k-max-tokens$", "", s)  # -8k-max-tokens
    s = re.sub(r"-max-tokens$", "", s)
    s = re.sub(r"-v01$", "", s)              # mixtral 뒤에 붙는거
    
    # 4. OpenAI 모델명 보정
    #    데이터셋 컬럼이 'openai-'를 달고 있으면 위에서 안 지워졌을 것임.
    #    혹시 지워졌거나 없는 경우(예: gpt-4o)를 대비해 보정
    if s.startswith("gpt-") or s.startswith("o3-") or s.startswith("o1-"):
        if not s.startswith("openai-"):
            s = "openai-" + s

    # 5. 가격표에 있는지 확인 (정확한 매칭 시도)
    if s in SPROUT_PRICE_1M:
        return s
    
    # 6. 그래도 없으면? (AWS Titan, Claude 등 특수 케이스 처리)
    #    Titan과 Claude는 버전 명시 차이로 실패할 수 있으므로 강제 매핑
    if "claude-3-5-sonnet" in s:
        return "claude-3-5-sonnet-v1"
    if "titan-text-premier" in s:
        return "titan-text-premier-v1"
        
    return None

# -----------------------------------------------------------------------------
# 4. 데이터 처리
# -----------------------------------------------------------------------------
print(f"🔄 Loading dataset: {DATASET_ID}...")
ds = load_dataset(DATASET_ID, split=SPLIT)

# 메타데이터 컬럼 제외하고 모델 컬럼만 추출
meta_cols = {"key", "dataset", "dataset_level", "dataset_idx", "prompt", "golden_answer"}
all_cols = list(ds.column_names)
model_cols = [c for c in all_cols if c not in meta_cols]

# 매핑 테이블 생성
col_to_price_map = {}
unmapped_cols = []

print("🔍 Mapping columns to price keys...")
for col in model_cols:
    price_key = match_price_key(col)
    if price_key:
        col_to_price_map[col] = price_key
        # 디버깅용 출력 (주석 해제시 확인 가능)
        # print(f"  ✅ '{col}' -> '{price_key}'")
    else:
        unmapped_cols.append(col)

print(f"  - Total model columns: {len(model_cols)}")
print(f"  - Successfully mapped: {len(col_to_price_map)}")
print(f"  - Unmapped: {len(unmapped_cols)}")

if unmapped_cols:
    print(f"⚠️ [WARN] Unmapped columns (will be skipped): {unmapped_cols}")
    # Titan, Claude가 여기 들어가면 안됨. 위 로직으로 해결될 것임.

# -----------------------------------------------------------------------------
# 5. Row 변환 및 비용 계산
# -----------------------------------------------------------------------------
rows = []
total_processed = 0

print("🔄 Processing rows...")

for i, ex in enumerate(ds):
    # 기본 정보
    row = {
        "sample_id": ex.get("key", i),
        "prompt": str(ex.get("prompt", "")),
        "eval_name": str(ex.get("dataset", "sprout")),
        "oracle_model_to_route_to": "",
    }

    # 모델별 데이터 처리
    for col_name, price_key in col_to_price_map.items():
        data = ex.get(col_name)
        
        # 데이터 유효성 체크
        if not isinstance(data, dict):
            continue
            
        score = data.get("score")
        n_in = data.get("num_input_tokens")
        n_out = data.get("num_output_tokens")
        
        if score is None or n_in is None or n_out is None:
            continue
            
        # 비용 계산 (1M 토큰 단위)
        p_in, p_out = SPROUT_PRICE_1M[price_key]
        cost = (n_in / 1_000_000.0 * p_in) + (n_out / 1_000_000.0 * p_out)
        
        # 결과 저장 (원본 컬럼명 사용)
        row[col_name] = float(score)
        row[f"{col_name}|total_cost"] = float(cost)
        
    rows.append(row)
    total_processed += 1

# -----------------------------------------------------------------------------
# 6. 저장
# -----------------------------------------------------------------------------
df = pd.DataFrame(rows)
df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

print(f"✅ 작업 완료!")
print(f"   - 저장 파일: {OUTPUT_CSV}")
print(f"   - 총 행 수: {len(df)}")
print(f"   - 사용된 모델 컬럼 수: {len(col_to_price_map)}")

# 확인용 출력 (처음 2줄의 일부 컬럼)
print("\n[Preview First 2 Rows (Transposed)]")
print(df.iloc[:2, :6].T)  # 앞부분 몇 개 컬럼만 확인

🔄 Loading dataset: CARROT-LLM-Routing/SPROUT-o3mini...
🔍 Mapping columns to price keys...
  - Total model columns: 14
  - Successfully mapped: 14
  - Unmapped: 0
🔄 Processing rows...
✅ 작업 완료!
   - 저장 파일: sprout_o3mini_routerbench_like.csv
   - 총 행 수: 30301
   - 사용된 모델 컬럼 수: 14

[Preview First 2 Rows (Transposed)]
                                                                           0  \
sample_id                  7f8b14630f644402d979fa215054d27ddd400bb1434d81...   
prompt                     If $200\%$ of $x$ is equal to $50\%$ of $y$, a...   
eval_name                                                 lighteval/MATH/all   
oracle_model_to_route_to                                                       
openai-o3-mini                                                           1.0   
openai-o3-mini|total_cost                                           0.001144   

                                                                           1  
sample_id                  70165ea235eb3bf992

In [13]:
from pathlib import Path
import pandas as pd

csv_files = [
    "embedllm_dataset.csv",
    "mixinstruct_dataset.csv",
    "routerbench_dataset.csv",
    "sprout_dataset.csv",
]

for name in csv_files:
    p = Path(name)
    if not p.exists():
        print(f"{name}\tMISSING")
        continue

    # 헤더 포함 CSV 기준: chunks로 메모리 과사용 방지
    reader = pd.read_csv(p, chunksize=200_000)
    first = next(reader)
    n_rows = len(first)
    n_cols = first.shape[1]

    for chunk in reader:
        n_rows += len(chunk)

    print(f"{name}\trows={n_rows}\tcols={n_cols}")


embedllm_dataset.csv	rows=29673	cols=228
mixinstruct_dataset.csv	rows=100000	cols=28
routerbench_dataset.csv	rows=36497	cols=26
sprout_dataset.csv	rows=30301	cols=32
